# 🔬 Intel Core Ultra NPU: Çizge Sinir Ağları (GNN) Performans Karakterizasyonu

Bu notebook, modern **Meteor Lake** mimarisindeki NPU yongasının, düzensiz ve bellek-yoğun iş yükleri olan **Çizge Sinir Ağları (GNN)** üzerindeki verimliliğini analiz etmek için tasarlanmıştır. 

## 📖 Çalışmanın Temel Soruları
1. **Memory Wall:** NPU'nun teorik hesaplama gücü, GNN'lerin düşük aritmetik yoğunluğu nedeniyle ne kadar kısıtlanıyor?
2. **Fusion Paradox:** Operatör birleştirme (fusion) her zaman performans artışı sağlar mı?
3. **Cross-Architecture:** GNN'lerin NPU uyumluluğu, ResNet (CNN) veya BERT (Transformer) ile kıyaslandığında ne durumda?

### 🛠️ 1. Ortam ve Donanım Tespiti
NPU cihazının OpenVINO tarafından tespiti ve projenin yapılandırılması.

In [ ]:
import os
import sys
import pandas as pd
import openvino as ov
from pathlib import Path
from IPython.display import Image, display, Markdown

# Donanım Bilgisi
core = ov.Core()
devices = core.available_devices
display(Markdown(f"**Tespit Edilen Cihazlar:** `{devices}`"))
if 'NPU' in devices:
    display(Markdown("<span style='color:green'>✅ NPU Aktif ve Kullanılabilir.</span>"))
else:
    display(Markdown("<span style='color:red'>⚠️ NPU tespit edilemedi! CPU üzerinde simülasyon yapılabilir.</span>"))

### 🏗️ 2. Model Havuzunun Hazırlanması
Tüm modellerin (GCN, GAT, ResNet50, BERT vb.) standardized FP32 ve INT8 formatlarına dönüştürülmesi.

In [ ]:
!$env:PYTHONIOENCODING='utf-8'; python scripts/generate_gnn_models.py

models = sorted(Path("models").glob("*.onnx"))
model_data = [{"Model": f.stem, "Size (MB)": round(f.stat().st_size / (1024*1024), 2)} for f in models]
pd.DataFrame(model_data)

### 🚀 3. Pipeline Yürütme (Benchmarking)
Seçilen bir model üzerinde derinlemesine profil çıkarma ve tüm modeller üzerinde ölçeklenebilirlik testi.

In [ ]:
TARGET_MODEL = "models/GCN_fp32.onnx"
ITERATIONS = 20

print(f"[RUNNING] Profiling and Scalability Pipeline for: {TARGET_MODEL}")
!python run_pipeline.py --iterations {ITERATIONS} --repeats 1 --profile-model {TARGET_MODEL}

### 📊 4. Akademik Analiz Paneli
Bu bölümde, pipeline tarafından üretilen akademik grafikler üzerinden sonuçları yorumlayacağız.

#### A. Roofline Modeli (NPU Limitleri)
GNN'lerin neden NPU'nun tam gücünü kullanamadığını (Memory-Bound bölgesi) gösterir.

In [ ]:
display(Image(filename="results/roofline_model.png"))

#### B. Latency Breakdown (%100 Stacked Bar)
Gecikmenin ne kadarının gerçek hesaplama (Compute), ne kadarının veri transferi (DMA) ve fırlatma (Dispatch) maliyeti olduğunu gösterir.

In [ ]:
display(Image(filename="results/latency_stacked_100pct.png"))

#### C. Pareto Frontier (Performans vs Karmaşıklık)
Model boyutu arttıkça gecikmenin nasıl ölçeklendiğini ve en verimli "Sweet Spot" noktasını tespit eder.

In [ ]:
display(Image(filename="results/pareto_frontier.png"))

### 📈 5. Performans Matrisi (Özet Tablo)
Tüm modellerin ham verilerini ve NPU hızlandırma oranlarını içeren karşılaştırmalı tablo.

In [ ]:
results_path = Path("results/scalability_matrix.csv")
if results_path.exists():
    df = pd.read_csv(results_path)
    # Güzelleştirilmiş tablo
    df_final = df[["model", "params_mil", "b_mean_ms", "o_mean_ms", "speedup", "ai"]].copy()
    df_final.columns = ["Model", "Param (M)", "Baseline (ms)", "NPU (ms)", "Speedup (x)", "AI (Flops/B)"]
    display(df_final.sort_values("Speedup (x)", ascending=False).style.background_gradient(subset=["Speedup (x)"], cmap="RdYlGn"))